<a href="https://colab.research.google.com/github/FJWangYantao/Pytorch-/blob/main/%E8%87%AA%E5%8A%A8%E5%BE%AE%E5%88%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# 图表设置
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\| **Autograd** \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

用 `torch.autograd` 进行自动微分
===============================================

训练神经网络时，**后向传播**是最常用的算法。在这个算法中参数（模型权重）通过损失函数在特定参数的**梯度**调整。

为计算梯度，PyTorch 提供了一个内置的 `torch.autograd`自动微分引擎。为任意计算图提供自动梯度计算。

考察最简单的单层神经网络，输入 `x` `w` `b` 和一些损失函数，可以在 Pytorch 中以下面方式定义：


In [10]:
import torch

x = torch.ones(5)  # 输入层
y = torch.zeros(3)  # 预期输出
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

张量，函数，计算图
==========================================

以上代码定义了如下**计算图**：

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

在这个网络中 `w` 和 `b` 都是需要被优化的**参数**。因此我们需要参照这些变量来计算梯度。Pytorch 设置了`requires_grad`这个属性。


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>创建张量时可以设置 <code>requires_grad</code> 这个参数, 或者创建后通过调用 <code>x.requires_grad_(True)</code> 这个方法</p>

</div>



用于构建计算图的函数实际上是一个`Function`的对象。这个对象能自动解析前向传播的方向和如何计算后向传播过程的导数。后向传播函数的引用被存储在张量的`grad_fn`属性中。可以在[这个文档](https://pytorch.org/docs/stable/autograd.html#function)中获取更多关于`Function`的信息。


In [11]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x7fb5e6806410>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7fb5e61d3f40>


计算梯度
===================

为优化神经网络中的权重，需要根据损失函数计算对应参数的导数，也就是说我们需要求出$\frac{\partial loss}{\partial w}$ 和 $\frac{\partial loss}{\partial b}$
为了计算这些导数我们调用 `loss.backward()` 再通过 `w.grad` 和d `b.grad` 取值


In [12]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.3218, 0.0971, 0.2748],
        [0.3218, 0.0971, 0.2748],
        [0.3218, 0.0971, 0.2748],
        [0.3218, 0.0971, 0.2748],
        [0.3218, 0.0971, 0.2748]])
tensor([0.3218, 0.0971, 0.2748])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>通过计算图的叶节点取得 <code>grad</code> 。 所求梯度的参数的<code>requires_grad</code> 参数被设为 <code>True</code>. 而其他图节点的参数不可获得。我们可以通过在给定的图上后向传播来进行梯度计算。如果我们需要用同一个计算图来进行多次反向传播，应该在backward参数上设置<code>retain_graph=True</code>否则重新运行会报错，因为反向传播后计算图就被释放了。</li>
</ul>
```

</div>



禁用梯度跟踪
===========================

默认情况下所有采用`requires_grad=True`的张量是可以追踪计算历史和支持梯度计算的。但在有些情况下，比如我们已经训练好模型了，想要测试它在一组输入上的表现，我们此时只想要模型的前向传播计算。所以通过`torch.no_grad()`来在特定一块代码禁用梯度跟踪


In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

用detach方法可以达到相同效果：

In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

有可能禁用梯度跟踪的原因：
- 神经网络中作为冻结层的参数。

- 加速前向传播计算速度，不跟踪梯度的张量计算更高效。


关于计算图
============================

概念上来说，自动微分保留了数据（张量）的记录和在一个有向无环图中的所有运算操作（包括形成新的张量）。在有向无环图中，叶节点是输入的张量，根节点是输出的张量。通过追踪计算图的根节点到叶节点，利用链式法则可以自动计算梯度


在一次前向传播过程中，自动微分同步做两件事：

- 执行所需的运算来计算结果张量
- 在有向无环图中保持运算符的*梯度函数*


后向传播在有向无环图的根节点的`.backward()`函数被调用时启动。`autograd`之后：

- 通过`.grad_fn`算出所有梯度
- 在对应张量的`.grad`属性中聚合这些梯度
- 通过链式法则传导到所有的叶节点

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>需要注意的点是计算图在每次<code>.backward()</code>被调用后重新创建。自动微分开始填充到新的计算图中。这使我们可以准确控制模型中数据流的状态。可以在每次迭代中根据需要改变张量的形状、大小、运算方式。</p>

</div>



张量梯度与雅可比乘积
========================================================

在许多例子中，我们使用标量损失函数，需要分别计算某些参数的梯度。有一些例外中输出的函数是一个不定形状的张量。在这种情况下Pytorch通过计算**雅可比乘积**而不是精确的梯度来运行

对函数 $\vec{y}=f(\vec{x})$, 有
$\vec{x}=\langle x_1,\dots,x_n\rangle$ 和
$\vec{y}=\langle y_1,\dots,y_m\rangle$,  $\vec{y}$ 对应 $\vec{x}$ 的梯度通过雅可比矩阵给出:

$$\begin{aligned}
J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
   \end{array}\right)
\end{aligned}$$

对于特定的输入$v=(v_1 \dots v_m)$，Pytorch允许计算雅可比乘积而非雅可比矩阵来获取梯度。 这通过把 $v$ 是做一个参数进行 `backward`操作。 $v$ 的大小需通原始张量大小相同，即对应我们希望计算乘积的张量。即只需要通过链式法则和雅可比乘积直接得到梯度结果，而无需计算庞大的雅可比矩阵:


In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

当对同一参数第二次调用`backward`时，梯度的值是不一样的。因为在做后向传播操作时，Pytorch同时聚合了梯度。比如计算得到的梯度数值被加到计算图的所有叶子节点的`grad`属性。如果想要获得正确的梯度，需要先把所有的`grad`属性置0。实际操作中，*optimizer*帮我们做了这个操作。


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>之前我们无参数调用了 <code>backward()</code> 。 本质上和调用了<code>backward(torch.tensor(1.0))</code>是一样的, 都是在标量函数下计算梯度的一种方法，比如神经网络训练过程中的损失函数。</p>

</div>



------------------------------------------------------------------------


Further Reading
===============

-   [Autograd
    Mechanics](https://pytorch.org/docs/stable/notes/autograd.html)
